In [65]:
import pandas as pd
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import numpy as np
import matplotlib.pyplot as plt 
from sklearn import datasets
from sklearn.cluster import KMeans

In [50]:
hospital_accounts = pd.read_csv('hospital_account_info.csv', index_col='Account_Num')
hospital_reinbursement = pd.read_csv('hospital_reimbursement.csv', index_col='Provider_Num')

In [51]:
hospital_accounts = hospital_accounts[['Facility Name', 'Address']]
hospital_accounts['Account_Num'] = hospital_accounts.index
hospital_accounts

,Facility Name,Address,Account_Num
Account_Num,,,
10605,SAGE MEMORIAL HOSPITAL,STATE ROUTE 264 SOUTH 191,10605
24250,WOODRIDGE BEHAVIORAL CENTER,600 NORTH 7TH STREET,24250
10341,DOUGLAS GARDENS HOSPITAL,5200 NE 2ND AVE,10341
81101,SUNCOAST BEHAVIORAL HEALTH CENTER,4480 51ST ST W,81101
39835,TREASURE VALLEY HOSPITAL,8800 WEST EMERALD STREET,39835
...,...,...,...
92281,JEWISH HOSPITAL - SHELBYVILLE,727 HOSPITAL DRIVE,92281
65248,DAYTON CHILDREN'S HOSPITAL,ONE CHILDRENS PLAZA,65248
92377,NORTH TEXAS STATE HOSPITAL,6515 KEMP BLVD,92377


In [52]:
hospital_reinbursement = hospital_reinbursement[['Provider Name', 'Provider Street Address']]
hospital_reinbursement['Provider_Num'] = hospital_reinbursement.index
hospital_reinbursement

,Provider Name,Provider Street Address,Provider_Num
Provider_Num,,,
839987,SOUTHEAST ALABAMA MEDICAL CENTER,1108 ROSS CLARK CIRCLE,839987
519118,MARSHALL MEDICAL CENTER SOUTH,2505 U S HIGHWAY 431 NORTH,519118
733073,ELIZA COFFEE MEMORIAL HOSPITAL,205 MARENGO STREET,733073
201752,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,201752
678488,ST VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,678488
...,...,...,...
403227,BAYLOR SCOTT & WHITE MEDICAL CENTER- COLLEGE S...,700 SCOTT & WHITE DRIVE,403227
797810,WALNUT HILL MEDICAL CENTER,7502 GREENVILLE AVENUE,797810
611900,"BAY AREA REGIONAL MEDICAL CENTER, LLC",200 BLOSSOM STREET,611900


In [53]:
hospital_reinbursement = hospital_reinbursement.rename(columns={'Provider Name':'Facility Name', 'Provider Street Address':'Address', 'Provider_Num':'Account_Num'})
hospital_reinbursement

,Facility Name,Address,Account_Num
Provider_Num,,,
839987,SOUTHEAST ALABAMA MEDICAL CENTER,1108 ROSS CLARK CIRCLE,839987
519118,MARSHALL MEDICAL CENTER SOUTH,2505 U S HIGHWAY 431 NORTH,519118
733073,ELIZA COFFEE MEMORIAL HOSPITAL,205 MARENGO STREET,733073
201752,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,201752
678488,ST VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,678488
...,...,...,...
403227,BAYLOR SCOTT & WHITE MEDICAL CENTER- COLLEGE S...,700 SCOTT & WHITE DRIVE,403227
797810,WALNUT HILL MEDICAL CENTER,7502 GREENVILLE AVENUE,797810
611900,"BAY AREA REGIONAL MEDICAL CENTER, LLC",200 BLOSSOM STREET,611900


In [54]:
vertical_concat = pd.concat([hospital_accounts, hospital_reinbursement], axis=0)
vertical_concat

,Facility Name,Address,Account_Num
10605,SAGE MEMORIAL HOSPITAL,STATE ROUTE 264 SOUTH 191,10605
24250,WOODRIDGE BEHAVIORAL CENTER,600 NORTH 7TH STREET,24250
10341,DOUGLAS GARDENS HOSPITAL,5200 NE 2ND AVE,10341
81101,SUNCOAST BEHAVIORAL HEALTH CENTER,4480 51ST ST W,81101
39835,TREASURE VALLEY HOSPITAL,8800 WEST EMERALD STREET,39835
...,...,...,...
403227,BAYLOR SCOTT & WHITE MEDICAL CENTER- COLLEGE S...,700 SCOTT & WHITE DRIVE,403227
797810,WALNUT HILL MEDICAL CENTER,7502 GREENVILLE AVENUE,797810
611900,"BAY AREA REGIONAL MEDICAL CENTER, LLC",200 BLOSSOM STREET,611900
693781,RESOLUTE HEALTH HOSPITAL,"555 CREEKSIDE XING,",693781


In [55]:
vertical_concat['Facility Name'] = vertical_concat['Facility Name'].apply(lambda x: x.split())
vertical_concat['Address'] = vertical_concat['Address'].apply(lambda x: x.split())
vertical_concat

,Facility Name,Address,Account_Num
10605,"[SAGE, MEMORIAL, HOSPITAL]","[STATE, ROUTE, 264, SOUTH, 191]",10605
24250,"[WOODRIDGE, BEHAVIORAL, CENTER]","[600, NORTH, 7TH, STREET]",24250
10341,"[DOUGLAS, GARDENS, HOSPITAL]","[5200, NE, 2ND, AVE]",10341
81101,"[SUNCOAST, BEHAVIORAL, HEALTH, CENTER]","[4480, 51ST, ST, W]",81101
39835,"[TREASURE, VALLEY, HOSPITAL]","[8800, WEST, EMERALD, STREET]",39835
...,...,...,...
403227,"[BAYLOR, SCOTT, &, WHITE, MEDICAL, CENTER-, CO...","[700, SCOTT, &, WHITE, DRIVE]",403227
797810,"[WALNUT, HILL, MEDICAL, CENTER]","[7502, GREENVILLE, AVENUE]",797810
611900,"[BAY, AREA, REGIONAL, MEDICAL, CENTER,, LLC]","[200, BLOSSOM, STREET]",611900
693781,"[RESOLUTE, HEALTH, HOSPITAL]","[555, CREEKSIDE, XING,]",693781


In [56]:
model_name = Word2Vec(sentences=vertical_concat['Facility Name'], vector_size=100, window=5, min_count=1, workers=4)

In [57]:
model_address = Word2Vec(sentences=vertical_concat['Address'], vector_size=100, window=5, min_count=1, workers=4)

In [58]:
model_name.wv.save_word2vec_format("name_embeddings.bin", binary=True)
model_address.wv.save_word2vec_format("address_embeddings.bin", binary=True)

In [59]:
name_vectors = KeyedVectors.load_word2vec_format("name_embeddings.bin", binary=True)
address_vectors = KeyedVectors.load_word2vec_format("address_embeddings.bin", binary=True)

In [60]:
name_vectors_list = []
for i in range(0,len(vertical_concat['Facility Name'])) :
    tokens = vertical_concat.iloc[i]['Facility Name']
    summed_vector = np.zeros(100)

    for token in tokens:
        if token in name_vectors:
            token_vector = name_vectors[token]
            summed_vector += token_vector
            
    name_vectors_list.append(summed_vector)
    

In [62]:
address_vectors_list = []
for i in range(0,len(vertical_concat['Address'])) :
    tokens = vertical_concat.iloc[i]['Address']
    summed_vector = np.zeros(100)

    for token in tokens:
        if token in name_vectors:
            token_vector = name_vectors[token]
            summed_vector += token_vector
            
    address_vectors_list.append(summed_vector)

In [63]:
vertical_concat['vectorized_name'] = name_vectors_list
vertical_concat['vectorized_address'] = address_vectors_list

In [64]:
vertical_concat

,Facility Name,Address,Account_Num,vectorized_name,vectorized_address
10605,"[SAGE, MEMORIAL, HOSPITAL]","[STATE, ROUTE, 264, SOUTH, 191]",10605,"[-0.5076035116799176, 0.5490385848097503, 0.21...","[-0.2620173841714859, 0.2882954031229019, 0.11..."
24250,"[WOODRIDGE, BEHAVIORAL, CENTER]","[600, NORTH, 7TH, STREET]",24250,"[-0.4612910018186085, 0.47819868044462055, 0.2...","[-0.14425934851169586, 0.17710216343402863, 0...."
10341,"[DOUGLAS, GARDENS, HOSPITAL]","[5200, NE, 2ND, AVE]",10341,"[-0.3557799104601145, 0.37960375286638737, 0.1...","[-4.4471496948972344e-05, 0.009999696165323257..."
81101,"[SUNCOAST, BEHAVIORAL, HEALTH, CENTER]","[4480, 51ST, ST, W]",81101,"[-0.7987435949034989, 0.8337352722883224, 0.33...","[-0.28151534497737885, 0.3182371184229851, 0.1..."
39835,"[TREASURE, VALLEY, HOSPITAL]","[8800, WEST, EMERALD, STREET]",39835,"[-0.4925671685487032, 0.5369903505779803, 0.23...","[-0.1791994869709015, 0.1706131547689438, 0.07..."
...,...,...,...,...,...
403227,"[BAYLOR, SCOTT, &, WHITE, MEDICAL, CENTER-, CO...","[700, SCOTT, &, WHITE, DRIVE]",403227,"[-1.0904594734311104, 1.2313539162278175, 0.48...","[-0.6053655594587326, 0.680865153670311, 0.283..."
797810,"[WALNUT, HILL, MEDICAL, CENTER]","[7502, GREENVILLE, AVENUE]",797810,"[-0.5255708256736398, 0.5855704583227634, 0.22...","[-0.027422234416007996, 0.024250549264252186, ..."
611900,"[BAY, AREA, REGIONAL, MEDICAL, CENTER,, LLC]","[200, BLOSSOM, STREET]",611900,"[-0.9063887968659401, 1.02201347053051, 0.3834...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
693781,"[RESOLUTE, HEALTH, HOSPITAL]","[555, CREEKSIDE, XING,]",693781,"[-0.6426272005774081, 0.7133091790601611, 0.28...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [66]:
kmeans_dataset = vertical_concat[['vectorized_name','vectorized_address']]

In [68]:
kmeans = KMeans(n_clusters=5339)
kmeans.fit(kmeans_dataset)
y_pred = kmeans.predict(kmeans_dataset)

ValueError: setting an array element with a sequence.

C'est très bancaleeeee